# Dementia QA — Human Evaluation UI (Caregiver QA Metrics)

Rate the quality of automatically generated question–answer pairs from dementia-care videos.

**Data sources** (auto-discovered from the repo):
- `Master/` — 10 videos × 2 approaches (`SingleAgent`, `MultiAgent-LLMChunking`)
- `Teepa/` — 15 videos × 4 approaches (`SingleAgent`, `DualAgent`, `MultiAgent-LLMChunking`, `RAG`)

**How to use:** run all cells top to bottom, then follow the on-screen steps:
1. Choose dataset(s)
2. Choose videos
3. Choose approaches + options (blind mode, shuffle, sampling)
4. Choose metrics, enter your name, and start rating

Progress is checkpointed after every submission, so you can close the notebook and resume later
(same annotator name). Results are saved as an Excel file in `eval/results/`.

Works locally (Jupyter/VS Code) and on Google Colab (mounts Drive — set `COLAB_BASE_DIR` below).

In [1]:
# Install required packages (safe to re-run; mostly needed on Colab / fresh envs)
import importlib.util, subprocess, sys

for pkg in ["ipywidgets", "pandas", "openpyxl"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("\u2713 Packages ready")

✓ Packages ready


In [2]:
# Imports + locate the repository (works locally and on Colab)
import os
import json
import time
import html
import random
from pathlib import Path

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

IN_COLAB = "google.colab" in sys.modules

# If you run this on Colab, upload/sync the repo to Drive and point this at it:
COLAB_BASE_DIR = "/content/drive/MyDrive/MedicalQA"

if IN_COLAB:
    from google.colab import drive, files
    drive.mount("/content/drive")
    BASE_DIR = Path(COLAB_BASE_DIR)
else:
    # Notebook lives in eval/ — look for Master/ or Teepa/ in cwd, then parents
    BASE_DIR = Path.cwd()
    for cand in [BASE_DIR, *BASE_DIR.parents]:
        if (cand / "Master").exists() or (cand / "Teepa").exists():
            BASE_DIR = cand
            break

OUTPUT_DIR = BASE_DIR / "eval" / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root:  {BASE_DIR}")
print(f"Results go: {OUTPUT_DIR}")
assert (BASE_DIR / "Master").exists() or (BASE_DIR / "Teepa").exists(), \
    "Could not find Master/ or Teepa/ - set BASE_DIR manually in this cell."


Repo root:  /Users/ruiweng/Lab/Medicine/MedicalQA
Results go: /Users/ruiweng/Lab/Medicine/MedicalQA/eval/results


In [ ]:
# Metric definitions - focused on caregiver-centered QA-pair quality
METRIC_DEFS = {
    "QA Alignment/Trustworthiness":
        "(Accurate, trustworthy, grounded in video.) The answer directly addresses the question, "
        "is supported by the source video, and does not add unsupported claims or contradict the video.",
    "QA Accessibility":
        "(Clear, fluent, easy to understand.) The QA pair is specific, readable, and accessible "
        "for caregivers, without unexplained clinical or technical language.",
    "QA Educational/Actionable Value":
        "(Actionable, useful, constructive information.) The QA pair teaches an important "
        "dementia-care idea and/or gives practical guidance a caregiver could apply.",
    "QA Mental Health Value":
        "(Resilience, empathy, mental health support.) The QA pair is constructive and "
        "emotionally supportive, promotes dignity and resilience, and avoids blame, fear, or judgment.",
}

# Which part of the QA pair each metric is judged on (controls highlighting in the UI)
METRIC_TYPES = {
    "q_only":  [],
    "a_only":  [],
    "qa_pair": list(METRIC_DEFS.keys()),
}

ALL_METRICS = list(METRIC_DEFS.keys())

METRIC_ATTRIBUTE_NAMES = {
    "QA Alignment/Trustworthiness": "Alignment/Trustworthiness",
    "QA Accessibility": "Accessibility",
    "QA Educational/Actionable Value": "Educational/Actionable Value",
    "QA Mental Health Value": "Mental Health Value",
}

METRIC_BINARY_QUESTIONS = {
    "QA Alignment/Trustworthiness": "Is the Q&A aligned with the video?",
    "QA Accessibility": "Is the Q&A easy for a caregiver to understand?",
    "QA Educational/Actionable Value": "Does the Q&A give useful or actionable guidance?",
    "QA Mental Health Value": "Is the Q&A supportive and safe for caregivers?",
}

METRIC_ATTRIBUTE_OPTIONS = {
    "QA Alignment/Trustworthiness": [
        "Excellent", "Good", "Fair", "Poor",
    ],
    "QA Accessibility": [
        "Very easy to understand", "Easy", "Somewhat difficult", "Difficult",
    ],
    "QA Educational/Actionable Value": [
        "Highly actionable", "Actionable", "Limited usefulness", "Not useful",
    ],
    "QA Mental Health Value": [
        "Highly supportive", "Supportive", "Limited support", "Unsupportive / potentially harmful",
    ],
}

METRIC_ERROR_OPTIONS = {
    "QA Alignment/Trustworthiness": [
        "Missing information", "Hallucination", "Contradiction", "Off-topic",
    ],
    "QA Accessibility": [
        "Difficult vocabulary", "Too long", "Ambiguous", "Poor organization",
    ],
    "QA Educational/Actionable Value": [
        "Not actionable", "Missing explanation", "Generic advice", "Incorrect recommendation",
    ],
    "QA Mental Health Value": [
        "Neutral / limited support", "Dismissive", "Potentially harmful",
    ],
}

METRIC_NO_ISSUE = {
    "QA Alignment/Trustworthiness": "No issue",
    "QA Accessibility": "No issue",
    "QA Educational/Actionable Value": "No issue",
    "QA Mental Health Value": "No issue",
}

# Store text labels exactly as annotators see them. Numeric mapping happens later.
NO_PROBLEM = "No issue"

BINARY_OPTIONS = ["Yes", "No"]
LEGACY_PROBLEM_VALUES = ["__needs_severity__", "Minor", "Moderate", "Severe"]

RECOMMENDATION_OPTIONS = [
    "Yes",
    "Yes, but with minor edits",
    "Only after major revisions",
    "No",
]

def binary_complete(value):
    return value in BINARY_OPTIONS


def binary_value(value):
    if not isinstance(value, dict):
        return None
    if binary_complete(value.get("binary")):
        return value.get("binary")
    if value.get("severity") == NO_PROBLEM:
        return "Yes"
    if value.get("severity") in LEGACY_PROBLEM_VALUES:
        return "No"
    return None


def recommendation_complete(value):
    return value in RECOMMENDATION_OPTIONS


def selected_errors(value):
    if isinstance(value, (list, tuple, set)):
        return [v for v in value if v]
    if isinstance(value, str) and value and value != NO_PROBLEM:
        return [v.strip() for v in value.split(";") if v.strip()]
    return []


def metric_complete(value):
    decision = binary_value(value)
    if not isinstance(value, dict) or not isinstance(value.get("attribute"), str) \
            or not value.get("attribute") or not binary_complete(decision):
        return False
    return decision == "Yes" or bool(selected_errors(value.get("errors", value.get("error"))))


def errors_text(value, metric):
    if not isinstance(value, dict):
        return ""
    if binary_value(value) == "Yes":
        return METRIC_NO_ISSUE[metric]
    return "; ".join(selected_errors(value.get("errors", value.get("error"))))


def qa_complete(qa):
    return all(metric_complete(qa["scores"].get(m)) for m in selected_metrics) \
        and recommendation_complete(qa.get("recommendation"))


print(f"✓ {len(ALL_METRICS)} metrics configured")

In [4]:
# Load every finalQA.json under Master/ and Teepa/
DATASETS = ["Master", "Teepa"]

all_qa_data = []      # every QA pair found on disk
filtered_qa_data = [] # the subset the annotator will rate
current_qa_index = 0
selected_datasets = []
selected_videos = []
selected_approaches = []
selected_metrics = ALL_METRICS[:]
annotator_name = ""
blind_mode = True


def load_all_qa():
    """Discover and load all QA files: <dataset>/<video>/<approach>/QA results/finalQA.json"""
    global all_qa_data
    all_qa_data = []
    for dataset in DATASETS:
        droot = BASE_DIR / dataset
        if not droot.exists():
            continue
        video_dirs = sorted(
            [d for d in droot.iterdir() if d.is_dir() and d.name.isdigit()],
            key=lambda d: int(d.name),
        )
        for vdir in video_dirs:
            for adir in sorted(d for d in vdir.iterdir() if d.is_dir()):
                qa_file = adir / "QA results" / "finalQA.json"
                if not qa_file.exists():
                    continue
                with open(qa_file, encoding="utf-8") as f:
                    items = json.load(f)
                for i, item in enumerate(items):
                    all_qa_data.append({
                        "uid": f"{dataset}_v{vdir.name}_{adir.name}_q{i + 1}",
                        "dataset": dataset,
                        "video": int(vdir.name),
                        "approach": adir.name,
                        "qa_num": i + 1,
                        "question": str(item.get("question", "")).strip(),
                        "answer": str(item.get("answer", "")).strip(),
                        "scores": {},
                        "recommendation": None,
                    })

    # Summary
    print(f"\u2713 Loaded {len(all_qa_data)} QA pairs total\n")
    summary = (
        pd.DataFrame(all_qa_data)
        .groupby(["dataset", "approach"])
        .agg(videos=("video", "nunique"), qa_pairs=("uid", "count"))
    )
    display(summary)


load_all_qa()

✓ Loaded 1587 QA pairs total



videos  qa_pairs
dataset approach                                
Master  MultiAgent-LLMChunking      10       200
        SingleAgent                 10       199
Teepa   DualAgent                   15       300
        MultiAgent-LLMChunking      15       300
        RAG                         15       300
        SingleAgent                 15       288

In [5]:
# Checkpointing — progress survives kernel restarts / closed tabs
def _checkpoint_path():
    safe = "".join(c if c.isalnum() or c in "-_" else "_" for c in (annotator_name or "anonymous"))
    return OUTPUT_DIR / f"checkpoint_{safe}.json"


def save_checkpoint():
    data = {}
    for qa in filtered_qa_data:
        if qa["scores"] or qa.get("recommendation") is not None:
            data[qa["uid"]] = {
                "scores": qa["scores"],
                "recommendation": qa.get("recommendation"),
            }
    with open(_checkpoint_path(), "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=1)


LEGACY_SEVERITY = {0: NO_PROBLEM, 1: "Minor", 2: "Moderate", 3: "Severe"}
LEGACY_RECOMMENDATION = {
    4: "Yes",
    3: "Yes, but with minor edits",
    2: "Only after major revisions",
    1: "No",
}


def normalize_recommendation(value):
    if recommendation_complete(value):
        return value
    return LEGACY_RECOMMENDATION.get(value)


def normalize_metric_state(metric, value):
    """Support older checkpoints that stored numeric mappings."""
    if isinstance(value, dict):
        state = dict(value)
        attr = state.get("attribute")
        if isinstance(attr, (int, float)) and int(attr) in {1, 2, 3, 4}:
            state["attribute"] = METRIC_ATTRIBUTE_OPTIONS[metric][4 - int(attr)]
        sev = state.get("severity")
        if isinstance(sev, (int, float)) and int(sev) in LEGACY_SEVERITY:
            state["severity"] = LEGACY_SEVERITY[int(sev)]
        if "binary" not in state:
            if state.get("severity") == NO_PROBLEM:
                state["binary"] = "Yes"
            elif state.get("severity") in LEGACY_PROBLEM_VALUES:
                state["binary"] = "No"
        if isinstance(state.get("error"), str) and "errors" not in state:
            state["errors"] = selected_errors(state["error"])
        return state
    if isinstance(value, (int, float)) and int(value) in LEGACY_SEVERITY:
        state = {"severity": LEGACY_SEVERITY[int(value)]}
        state["binary"] = "Yes" if int(value) == 0 else "No"
        if int(value) == 0:
            state["error"] = METRIC_NO_ISSUE[metric]
        return state
    return {}


def restore_checkpoint():
    """Merge previously saved labels into the current selection. Returns #restored."""
    path = _checkpoint_path()
    if not path.exists():
        return 0
    with open(path, encoding="utf-8") as f:
        saved = json.load(f)
    n = 0
    for qa in filtered_qa_data:
        raw = saved.get(qa["uid"])
        if not raw:
            continue
        saved_scores = raw.get("scores", raw) if isinstance(raw, dict) else {}
        for metric, value in saved_scores.items():
            if metric in ALL_METRICS:
                qa["scores"][metric] = normalize_metric_state(metric, value)
        if isinstance(raw, dict):
            qa["recommendation"] = normalize_recommendation(raw.get("recommendation"))
        n += 1
    return n


def fmt_text(text):
    """Escape HTML and preserve line breaks for display."""
    return html.escape(text).replace("\n", "<br>")

print("\u2713 Helpers ready")

✓ Helpers ready


In [6]:
# Step 1 — dataset selection
def show_dataset_selection():
    clear_output(wait=True)
    print("=" * 80)
    print("DEMENTIA QA EVALUATION \u2014 EDUCATIONAL VALUE")
    print("=" * 80)
    print("\nStep 1/4: Choose dataset(s)\n")

    available = sorted({qa["dataset"] for qa in all_qa_data})
    options = [[d] for d in available]
    if len(available) > 1:
        options.append(available)  # "Both"

    def on_click(button):
        global selected_datasets
        selected_datasets = button._datasets
        show_video_selection()

    buttons = []
    for opt in options:
        label = " + ".join(opt) if len(opt) > 1 else opt[0]
        n = sum(1 for qa in all_qa_data if qa["dataset"] in opt)
        btn = widgets.Button(
            description=f"{label}  ({n} QAs)",
            button_style="info",
            layout=widgets.Layout(width="240px", height="50px"),
        )
        btn._datasets = opt
        btn.on_click(on_click)
        buttons.append(btn)

    display(widgets.HBox(buttons))

In [7]:
# Step 2 — video selection
def show_video_selection():
    clear_output(wait=True)
    print("=" * 80)
    print(f"Dataset(s): {', '.join(selected_datasets)}")
    print("=" * 80)
    print("\nStep 2/4: Select videos (all selected by default)\n")

    pool = [qa for qa in all_qa_data if qa["dataset"] in selected_datasets]
    video_keys = sorted({(qa["dataset"], qa["video"]) for qa in pool})

    checkboxes = []
    for ds, vid in video_keys:
        n = sum(1 for qa in pool if qa["dataset"] == ds and qa["video"] == vid)
        cb = widgets.Checkbox(
            value=True,
            description=f"{ds} \u2014 video {vid} ({n})",
            indent=False,
            layout=widgets.Layout(width="240px"),
        )
        cb._key = (ds, vid)
        checkboxes.append(cb)

    def set_all(value):
        for cb in checkboxes:
            cb.value = value

    btn_all = widgets.Button(description="Select all", layout=widgets.Layout(width="110px"))
    btn_none = widgets.Button(description="Select none", layout=widgets.Layout(width="110px"))
    btn_all.on_click(lambda b: set_all(True))
    btn_none.on_click(lambda b: set_all(False))

    next_button = widgets.Button(
        description="Next: Approaches \u2192",
        button_style="success",
        layout=widgets.Layout(width="200px", height="40px"),
    )
    status = widgets.HTML("<i>Select at least one video and click Next</i>")

    def on_next(b):
        global selected_videos
        selected_videos = [cb._key for cb in checkboxes if cb.value]
        if not selected_videos:
            status.value = "<span style='color:red;'>\u26a0 Please select at least one video</span>"
            return
        show_approach_selection()

    next_button.on_click(on_next)

    grid = widgets.GridBox(
        checkboxes,
        layout=widgets.Layout(grid_template_columns="repeat(3, 260px)"),
    )
    display(widgets.VBox([
        widgets.HBox([btn_all, btn_none]),
        grid,
        next_button,
        status,
    ]))

In [8]:
# Step 3 — approach selection + options (blind mode, shuffle, sampling)
def show_approach_selection():
    clear_output(wait=True)
    print("=" * 80)
    print(f"Dataset(s): {', '.join(selected_datasets)} | Videos: {len(selected_videos)}")
    print("=" * 80)
    print("\nStep 3/4: Select approaches + options\n")

    pool = [qa for qa in all_qa_data
            if (qa["dataset"], qa["video"]) in selected_videos]
    approaches = sorted({qa["approach"] for qa in pool})

    checkboxes = []
    for app in approaches:
        n = sum(1 for qa in pool if qa["approach"] == app)
        cb = widgets.Checkbox(
            value=True,
            description=f"{app} ({n} QAs)",
            indent=False,
            layout=widgets.Layout(width="320px"),
        )
        cb._approach = app
        checkboxes.append(cb)

    blind_cb = widgets.Checkbox(
        value=True, indent=False,
        description="Blind mode \u2014 hide approach names while rating (recommended)",
        layout=widgets.Layout(width="500px"),
    )
    shuffle_cb = widgets.Checkbox(
        value=True, indent=False,
        description="Shuffle QA order (recommended, avoids order bias)",
        layout=widgets.Layout(width="500px"),
    )
    seed_input = widgets.IntText(value=42, description="Seed:", layout=widgets.Layout(width="180px"))
    sample_input = widgets.IntText(
        value=0, description="Sample:", layout=widgets.Layout(width="180px"),
    )
    sample_note = widgets.HTML(
        "<i>QAs per approach per video (0 = all). Sampling uses the seed, so all annotators "
        "with the same seed rate the same subset.</i>"
    )

    next_button = widgets.Button(
        description="Next: Metrics \u2192",
        button_style="success",
        layout=widgets.Layout(width="200px", height="40px"),
    )
    status = widgets.HTML("<i>Select at least one approach and click Next</i>")

    def on_next(b):
        global selected_approaches, filtered_qa_data, blind_mode
        selected_approaches = [cb._approach for cb in checkboxes if cb.value]
        if not selected_approaches:
            status.value = "<span style='color:red;'>\u26a0 Please select at least one approach</span>"
            return
        blind_mode = blind_cb.value
        seed = seed_input.value
        per_group = max(0, sample_input.value)

        chosen = [qa for qa in pool if qa["approach"] in selected_approaches]

        # Deterministic sampling per (dataset, video, approach) group
        if per_group:
            rng = random.Random(seed)
            groups = {}
            for qa in chosen:
                groups.setdefault((qa["dataset"], qa["video"], qa["approach"]), []).append(qa)
            chosen = []
            for key in sorted(groups):
                items = groups[key]
                if len(items) > per_group:
                    items = rng.sample(items, per_group)
                    items.sort(key=lambda q: q["qa_num"])
                chosen.extend(items)

        if shuffle_cb.value:
            random.Random(seed).shuffle(chosen)
        else:
            chosen.sort(key=lambda q: (q["dataset"], q["video"], q["qa_num"], q["approach"]))

        filtered_qa_data = chosen
        show_metric_selection()

    next_button.on_click(on_next)

    display(widgets.VBox([
        widgets.HTML("<b>Approaches:</b>"),
        widgets.VBox(checkboxes),
        widgets.HTML("<br><b>Options:</b>"),
        blind_cb,
        shuffle_cb,
        widgets.HBox([seed_input, sample_input]),
        sample_note,
        widgets.HTML("<br>"),
        next_button,
        status,
    ]))

In [ ]:
# Step 4 — metric selection + annotator name
def show_metric_selection():
    clear_output(wait=True)
    print("=" * 80)
    print(f"Dataset(s): {', '.join(selected_datasets)} | Videos: {len(selected_videos)} "
          f"| Approaches: {', '.join(selected_approaches)}")
    print(f"QA pairs to evaluate: {len(filtered_qa_data)}")
    print("=" * 80)
    print("\nStep 4/4: Select metrics and enter your name\n")

    checkboxes = []
    for metric in ALL_METRICS:
        cb = widgets.Checkbox(
            value=True, description=metric, indent=False,
            layout=widgets.Layout(width="320px"),
        )
        checkboxes.append(cb)

    name_input = widgets.Text(
        value="", placeholder="Enter your name",
        description="Annotator:", layout=widgets.Layout(width="400px"),
    )
    start_button = widgets.Button(
        description="\u2713 Start Evaluation",
        button_style="success",
        layout=widgets.Layout(width="200px", height="40px"),
    )
    status = widgets.HTML("<i>Select metrics and click Start</i>")

    def on_start(b):
        global selected_metrics, current_qa_index, annotator_name
        selected_metrics = [cb.description for cb in checkboxes if cb.value]
        annotator_name = name_input.value.strip()
        if not selected_metrics:
            status.value = "<span style='color:red;'>\u26a0 Please select at least one metric</span>"
            return

        restored = restore_checkpoint()
        # Jump to the first QA that still has unrated selected metrics or recommendation
        current_qa_index = 0
        for i, qa in enumerate(filtered_qa_data):
            if not qa_complete(qa):
                current_qa_index = i
                break
        else:
            current_qa_index = len(filtered_qa_data)

        if restored:
            print(f"\n\u2713 Resumed checkpoint: {restored} QA pairs already have labels; "
                  f"continuing at QA {current_qa_index + 1}")
            time.sleep(1.5)
        show_qa_evaluation()

    start_button.on_click(on_start)

    display(widgets.VBox([
        widgets.HTML("<b>Metrics to evaluate (all selected by default):</b>"),
        widgets.VBox(checkboxes),
        widgets.HTML("<br>"),
        name_input,
        start_button,
        status,
    ]))

In [ ]:
# Evaluation screen
def show_qa_evaluation():
    global current_qa_index

    if current_qa_index >= len(filtered_qa_data):
        save_results()
        return

    clear_output(wait=True)
    qa = filtered_qa_data[current_qa_index]
    total = len(filtered_qa_data)
    done = sum(1 for q in filtered_qa_data if qa_complete(q))

    approach_label = "(hidden — blind mode)" if blind_mode else qa["approach"]
    print("=" * 80)
    print(f"QA {current_qa_index + 1} of {total}   |   fully rated so far: {done}")
    print("=" * 80)
    print(f"Dataset: {qa['dataset']} | Video: {qa['video']} | Approach: {approach_label}")
    print("=" * 80)

    display(HTML(f"""
    <div style="background:#fff3cd; padding:14px; margin:10px 0; border-radius:8px;
                border-left:4px solid #ff9800;">
        <strong style="color:#856404;">Question</strong><br>
        <span style="color:#333; font-size:15px;">{fmt_text(qa['question'])}</span>
    </div>
    <div style="background:#d4edda; padding:14px; margin:10px 0; border-radius:8px;
                border-left:4px solid #28a745; max-height:400px; overflow-y:auto;">
        <strong style="color:#155724;">Answer</strong><br>
        <span style="color:#333; font-size:15px;">{fmt_text(qa['answer'])}</span>
    </div>
    """))

    tag_colors = {"q_only": ("#856404", "Question"), "a_only": ("#155724", "Answer"),
                  "qa_pair": ("#2c5aa0", "Q&A pair")}

    metric_widgets = {}
    metric_rows = []
    for metric in selected_metrics:
        mtype = next(t for t, ms in METRIC_TYPES.items() if metric in ms)
        color, tag = tag_colors[mtype]
        existing = qa["scores"].get(metric, {})
        if not isinstance(existing, dict):
            existing = normalize_metric_state(metric, existing)
        decision_value = binary_value(existing)
        attribute_value = existing.get("attribute") if existing.get("attribute") in METRIC_ATTRIBUTE_OPTIONS[metric] else None
        error_value = tuple(e for e in selected_errors(existing.get("errors", existing.get("error")))
                            if e in METRIC_ERROR_OPTIONS[metric])

        header = widgets.HTML(f"""
        <div style="background:#f5f5f5; padding:10px 14px; margin-top:14px; border-radius:8px;
                    border-left:4px solid #4da6ff;">
            <b style="color:#2c5aa0; font-size:15px;">{metric}</b>
            <span style="background:{color}; color:white; border-radius:10px; padding:1px 8px;
                         font-size:11px; margin-left:8px;">{tag}</span><br>
            <span style="color:#555; font-style:italic; font-size:13px;">{METRIC_DEFS[metric]}</span>
        </div>""")
        attr_name = METRIC_ATTRIBUTE_NAMES[metric]
        attribute_label = widgets.HTML(f"<b style='margin-left:10px;'>{attr_name}</b>")
        attribute = widgets.RadioButtons(
            options=METRIC_ATTRIBUTE_OPTIONS[metric],
            value=attribute_value,
            layout=widgets.Layout(width="700px", margin="2px 0 0 24px"),
        )
        error_label = widgets.HTML("<b style='margin-left:10px;'>If No, what is wrong? Select all that apply.</b>")
        error = widgets.SelectMultiple(
            options=METRIC_ERROR_OPTIONS[metric],
            value=error_value,
            rows=min(4, len(METRIC_ERROR_OPTIONS[metric])),
            layout=widgets.Layout(width="700px", margin="2px 0 0 24px",
                                  display="block" if decision_value == "No" else "none"),
        )
        error_label.layout.display = "block" if decision_value == "No" else "none"
        binary_label = widgets.HTML(f"<b style='margin-left:10px;'>{METRIC_BINARY_QUESTIONS[metric]}</b>")
        binary = widgets.RadioButtons(
            options=BINARY_OPTIONS,
            value=decision_value,
            layout=widgets.Layout(width="700px", margin="2px 0 0 24px"),
        )

        def on_binary_change(change, err=error, err_label=error_label):
            if change["new"] == "No":
                err.layout.display = "block"
                err_label.layout.display = "block"
            else:
                err.value = ()
                err.layout.display = "none"
                err_label.layout.display = "none"

        binary.observe(on_binary_change, names="value")
        metric_widgets[metric] = {
            "attribute": attribute, "error": error, "binary": binary,
        }
        metric_rows += [header, attribute_label, attribute,
                        binary_label, binary, error_label, error]

    recommendation = widgets.RadioButtons(
        options=RECOMMENDATION_OPTIONS,
        value=qa.get("recommendation") if recommendation_complete(qa.get("recommendation")) else None,
        layout=widgets.Layout(width="700px", margin="2px 0 0 24px"),
    )
    recommendation_header = widgets.HTML(
        "<div style='background:#f5f5f5; padding:10px 14px; margin-top:14px; border-radius:8px; "
        "border-left:4px solid #2c5aa0;'><b style='color:#2c5aa0; font-size:15px;'>"
        "Would you confidently recommend this Q&A to a caregiver?</b><br>"
        "<span style='color:#555; font-style:italic; font-size:13px;'>"
        "Final caregiver-facing judgment for the whole QA pair.</span></div>"
    )

    btn_prev = widgets.Button(
        description="← Previous", button_style="info",
        layout=widgets.Layout(width="150px", height="40px"),
        disabled=(current_qa_index == 0),
    )
    btn_next = widgets.Button(
        description="Submit & Next →", button_style="success",
        layout=widgets.Layout(width="180px", height="40px"),
    )
    btn_save = widgets.Button(
        description="💾 Save & Exit", button_style="warning",
        layout=widgets.Layout(width="150px", height="40px"),
    )
    status = widgets.HTML("")

    def store_scores():
        for metric, controls in metric_widgets.items():
            binary = controls["binary"]
            state = {
                "attribute": controls["attribute"].value,
                "errors": list(controls["error"].value),
                "binary": binary.value,
            }
            if metric_complete(state):
                qa["scores"][metric] = state
            else:
                qa["scores"].pop(metric, None)
        qa["recommendation"] = recommendation.value if recommendation_complete(recommendation.value) else None
        save_checkpoint()

    def missing_metrics():
        missing = []
        for metric, controls in metric_widgets.items():
            parts = []
            if controls["attribute"].value is None:
                parts.append("attribute")
            if controls["binary"].value is None:
                parts.append("Yes/No question")
            elif controls["binary"].value == "No":
                if not controls["error"].value:
                    parts.append("problem type")
            if parts:
                missing.append(f"{metric} ({', '.join(parts)})")
        if recommendation.value is None:
            missing.append("caregiver recommendation")
        return missing

    def on_next(b):
        global current_qa_index
        missing = missing_metrics()
        if missing:
            status.value = ("<span style='color:red;'>⚠ Please complete: "
                            + ", ".join(missing) + "</span>")
            return
        store_scores()
        current_qa_index += 1
        show_qa_evaluation()

    def on_prev(b):
        global current_qa_index
        store_scores()
        current_qa_index -= 1
        show_qa_evaluation()

    def on_save(b):
        store_scores()
        save_results(partial=True)

    btn_next.on_click(on_next)
    btn_prev.on_click(on_prev)
    btn_save.on_click(on_save)

    display(widgets.VBox(metric_rows + [recommendation_header, recommendation]))
    display(widgets.HBox([btn_prev, btn_next, btn_save],
                         layout=widgets.Layout(margin="20px 0")))
    display(status)

In [ ]:
# Save results to Excel
def save_results(partial=False):
    clear_output(wait=True)
    print("=" * 80)
    print("SAVING RESULTS" + (" (partial \u2014 you can resume later)" if partial else ""))
    print("=" * 80)

    rows = []
    for qa in filtered_qa_data:
        row = {
            "Dataset": qa["dataset"],
            "Video Index": qa["video"],
            "Approach": qa["approach"],
            "QA ID": qa["uid"],
            "Question": qa["question"],
            "Answer": qa["answer"],
        }
        for metric in ALL_METRICS:
            state = qa["scores"].get(metric, {})
            if not isinstance(state, dict):
                state = normalize_metric_state(metric, state)
            row[f"{metric} Attribute"] = state.get("attribute", "")
            row[f"{metric} Error Type"] = errors_text(state, metric)
            row[f"{metric} Yes/No"] = binary_value(state) or ""
        recommendation = qa.get("recommendation")
        row["Caregiver Recommendation"] = recommendation if recommendation_complete(recommendation) else ""
        row["Annotator"] = annotator_name
        rows.append(row)

    df = pd.DataFrame(rows)

    timestamp = time.strftime("%Y%m%d_%H%M%S")
    name_part = annotator_name if annotator_name else "anonymous"
    ds_part = "-".join(selected_datasets)
    filename = f"{ds_part}_{name_part}_Eval_{timestamp}.xlsx"
    out_path = OUTPUT_DIR / filename
    df.to_excel(out_path, index=False, engine="openpyxl")

    rated = sum(1 for qa in filtered_qa_data if qa_complete(qa))
    print(f"\n\u2713 Saved to: {out_path}")
    print(f"\u2713 QA pairs fully rated: {rated} / {len(filtered_qa_data)}")
    print(f"\u2713 Annotator: {annotator_name or '(anonymous)'}")
    print(f"\u2713 Metrics: {', '.join(selected_metrics)}")
    if partial:
        print("\n\u23f8 Progress is checkpointed. To resume: re-run the start cell, make the "
              "same selections, and enter the same annotator name.")
    else:
        print("\n\U0001f389 Evaluation complete \u2014 thank you!")

    if IN_COLAB:
        try:
            files.download(str(out_path))
            print("\u2713 Download started")
        except Exception as e:
            print(f"\u26a0 Auto-download failed ({e}); download it from the Files panel.")

    binary_cols = [f"{m} Yes/No" for m in selected_metrics]
    rec_col = "Caregiver Recommendation"
    scored = df[df[binary_cols + [rec_col]].replace("", pd.NA).notna().any(axis=1)] if binary_cols else df
    if len(scored):
        print("\n" + "=" * 80)
        print("DESCRIPTIVE SUMMARY PER APPROACH")
        print("Text labels only; apply score/code mapping later during analysis.")
        print("=" * 80)
        display(scored.groupby("Approach").size().rename("rated_pairs").to_frame())

    display(df.head(10))

In [ ]:
# \u25b6 START — run this cell to begin (re-run it to start over)
show_dataset_selection()

---
## Optional: aggregate results across annotators

Run the cell below after collecting result files in `eval/results/` for quick pilot
summaries: Yes/No decisions by approach, error-taxonomy counts, caregiver recommendation,
and attribute-label distributions. It reads both the
`.xlsx` files this notebook writes and the `.csv` files downloaded from the web
UI (`eval/web/`). Treat these as descriptive pilot summaries; calculate formal
inter-annotator agreement separately on the shared QA pairs.


In [ ]:
# Aggregate every result file in eval/results/ across annotators.
# Picks up both the notebook's .xlsx exports and the CSVs downloaded from the
# web UI (eval/web/) — they share the same text-label column names.
result_files = sorted(OUTPUT_DIR.glob("*_Eval_*.xlsx")) + sorted(OUTPUT_DIR.glob("*.csv"))
if not result_files:
    print("No result files found in", OUTPUT_DIR)
else:
    frames = []
    for f in result_files:
        d = pd.read_csv(f) if f.suffix.lower() == ".csv" else pd.read_excel(f)
        d["__file"] = f.name
        frames.append(d)
    combined = pd.concat(frames, ignore_index=True)

    binary_cols = [f"{m} Yes/No" for m in ALL_METRICS if f"{m} Yes/No" in combined.columns]
    attribute_cols = [f"{m} Attribute" for m in ALL_METRICS if f"{m} Attribute" in combined.columns]
    error_cols = [f"{m} Error Type" for m in ALL_METRICS if f"{m} Error Type" in combined.columns]
    rec_col = "Caregiver Recommendation"
    rating_subset = binary_cols + ([rec_col] if rec_col in combined.columns else [])
    rated = combined[combined[rating_subset].replace("", pd.NA).notna().any(axis=1)] if rating_subset else combined

    print(f"Files: {len(result_files)} | Rated rows: {len(rated)} | "
          f"Annotators: {rated['Annotator'].nunique()}")

    print("\nRated pairs per approach:")
    display(rated.groupby("Approach").size().rename("rated_pairs").to_frame())

    if attribute_cols:
        print("\nAttribute label counts by approach:")
        for col in attribute_cols:
            metric = col.replace(" Attribute", "")
            print(f"\n{metric}")
            display(rated.groupby(["Approach", col]).size().rename("n").reset_index()
                    .sort_values(["Approach", "n"], ascending=[True, False]))

    if binary_cols:
        print("\nYes/No counts by approach:")
        for col in binary_cols:
            metric = col.replace(" Yes/No", "")
            print(f"\n{metric}")
            display(rated.groupby(["Approach", col]).size().rename("n").reset_index()
                    .sort_values(["Approach", "n"], ascending=[True, False]))

    if rec_col in rated.columns:
        print("\nCaregiver recommendation counts by approach:")
        display(rated.groupby(["Approach", rec_col]).size().rename("n").reset_index()
                .sort_values(["Approach", "n"], ascending=[True, False]))

    if error_cols:
        print("\nProblem taxonomy counts by approach:")
        for col in error_cols:
            metric = col.replace(" Error Type", "")
            print(f"\n{metric}")
            display(rated.groupby(["Approach", col]).size().rename("n").reset_index()
                    .sort_values(["Approach", "n"], ascending=[True, False]))

    overlap = rated.groupby("QA ID")["Annotator"].nunique()
    shared = overlap[overlap > 1].index
    if len(shared):
        print(f"\n{len(shared)} QA pairs rated by 2+ annotators. Use these for formal IAA.")
        print("Apply the agreed text-to-score mapping later before numeric agreement/correlation.")
    else:
        print("\nNo QA pair was rated by more than one annotator — formal IAA cannot be computed yet.")
